# Graphiko: Business Cluster Channel Descriptions

Connects to MongoDB, finds the latest **business** cluster in the `finder` DB context,
and prints channel descriptions for channels in that cluster.


In [ ]:
# Install dependencies (Colab-friendly)
!pip install -q pymongo dnspython


In [ ]:
from pymongo import MongoClient
from google.colab import userdata

try:
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(f'An error occurred while connecting: {e}')
    raise


In [ ]:
# Finder DB collections
# Reusing the same cluster lookup approach as the existing Grafiko notebook.
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']

# 1) Latest cluster version
latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None
print(f'Latest cluster version: {latest_version}')

# 2) Business cluster in latest version
business_cluster_id = None
if latest_version is None:
    print('No clusters found.')
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })
    if business_cluster:
        business_cluster_id = business_cluster['_id']
        print(f"Business cluster id: {business_cluster_id}")
        print(f"Business cluster name: {business_cluster.get('name')}")
    else:
        print('No business cluster found in latest version.')


In [ ]:
# 3) Get channel descriptions for channels in the business cluster and print them
if business_cluster_id is None:
    print('Cannot fetch channel descriptions without a business cluster id.')
else:
    # Channel IDs mapped to this cluster
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs if d.get('textId')]

    # Fetch channel metadata + descriptions
    channel_docs = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1, 'description': 1}
    ))

    print(f'Channels in business cluster: {len(channel_docs)}')
    print('=' * 100)
    for i, ch in enumerate(channel_docs, start=1):
        title = ch.get('title') or 'Unknown title'
        desc = ch.get('description') or '<no description>'
        print(f"{i}. {title} ({ch.get('channelId')})")
        print(desc)
        print('-' * 100)
